In [0]:
select count(distinct patient_id) from com_edp_prd.com_raw.kom_medical_events
where DIAGNOSIS_CODES like '%E763%' 
and SERVICE_DATE between '2020-08-01' and '2025-07-31'

In [0]:
CREATE OR REPLACE TEMPORARY VIEW MPSII_1Dx_Unspecified AS 
(
  SELECT * FROM (
    SELECT DISTINCT 
      PATIENT_ID,
      COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
  ) AS combined
  WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31'
);

In [0]:
SELECT DISTINCT PATIENT_ID AS PATIENT_ID 

FROM 

(

SELECT PATIENT_ID,  COUNT(DISTINCT FILL_DATE) AS NUMBER_OF_CLAIMS

FROM MPSII_1Dx_Unspecified--TABLE

--WHERE ARRAYS_OVERLAP (SPLIT(DIAGNOSIS_CODES, '|'), ARRAY_CONSTRUCT_COMPACT('E761'))

--WHERE FILL_DATE BETWEEN '2020-04-01' AND '2025-03-31' 

GROUP BY PATIENT_ID

)

WHERE NUMBER_OF_CLAIMS >=2
 

In [0]:
-- Patients with >=2 E763 claims and 0 E761 claims
WITH E763_CLAIMS AS (
    SELECT
        PATIENT_ID,
        COUNT(DISTINCT SERVICE_DATE) AS E763_COUNT
    FROM com_edp_prd.com_raw.kom_medical_events

    WHERE DIAGNOSIS_CODES LIKE '%E763%'   -- check for unspecified code
    GROUP BY PATIENT_ID
),
E761_CLAIMS AS (
    SELECT DISTINCT
        PATIENT_ID
    FROM com_edp_prd.com_raw.kom_medical_events

    WHERE DIAGNOSIS_CODES LIKE '%E761%'   -- check for specified code
)
 
SELECT DISTINCT
    E763.PATIENT_ID
FROM E763_CLAIMS E763
LEFT JOIN E761_CLAIMS E761
    ON E763.PATIENT_ID = E761.PATIENT_ID
WHERE
    E763.E763_COUNT >= 2       -- ≥2 E763 claims
    AND E761.PATIENT_ID IS NULL;  -- exclude anyone with E761

In [0]:
-- 1) Patients with >=2 E763 diagnosis claims (medical table)

WITH E763_PATIENTS AS (

    SELECT

        PATIENT_ID,

        COUNT(DISTINCT SERVICE_DATE) AS E763_CLAIM_COUNT

    FROM com_edp_prd.com_raw.kom_medical_events


    WHERE DIAGNOSIS_CODES LIKE '%E763%'       -- Unspecified MPS II Dx

    GROUP BY PATIENT_ID

),
 
-- 2) Patients with at least 1 treatment code (procedure or NDC11)

TREATMENT_PATIENTS AS (

    -- From <MEDICAL_EVENTS>: procedure or NDC11

    SELECT DISTINCT

        PATIENT_ID

    FROM com_edp_prd.com_raw.kom_medical_events


    WHERE

        PROCEDURE_CODE IN ('J1743')                 -- Procedure code(s)

        OR NDC11 IN ('54092070001', '540920700')    -- NDC11 codes
 
    UNION
 
    -- From PHARMACY_EVENTS: NDC11 only

    SELECT DISTINCT

        PATIENT_ID

    FROM com_edp_prd.com_raw.kom_pharmacy_events


    WHERE

        NDC11 IN ('54092070001', '540920700')       -- Same NDC11 set

)
 
-- 3) Final: Patients with >=2 E763 claims AND at least 1 treatment code

SELECT

    COUNT(DISTINCT E.PATIENT_ID) AS PATIENT_COUNT

FROM E763_PATIENTS E

JOIN TREATMENT_PATIENTS T

    ON E.PATIENT_ID = T.PATIENT_ID

WHERE

    E.E763_CLAIM_COUNT >= 2;

 

In [0]:
-- Step 1: Identify patients with E763 >=2 and E761 = 0 
-- using SERVICE_DATE (Diagnosis window: Aug 2020 – Jul 2025)
 
WITH DX_COUNTS AS (
    SELECT
        PATIENT_ID,
 
        -- Count of E763 diagnosis events within 5-year window
        COUNT(DISTINCT CASE
                           WHEN DIAGNOSIS_CODES LIKE '%E763%'
                                AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                           THEN SERVICE_DATE
                       END) AS E763_COUNT,
 
        -- Count of E761 diagnosis events within 5-year window
        COUNT(DISTINCT CASE
                           WHEN DIAGNOSIS_CODES LIKE '%E761%'
                                AND SERVICE_DATE BETWEEN '2020-08-01' AND '2025-07-31'
                           THEN SERVICE_DATE
                       END) AS E761_COUNT
 
    FROM com_edp_prd.com_raw.kom_medical_events
    GROUP BY PATIENT_ID
),
 
-- Step 2: Patients with ≥1 treatment code (procedure or NDC11)
-- Treatment window: Aug 2023 – Jul 2025
 
TREATMENT_PTS AS (
 
    -- Treatment in MEDICAL EVENTS table
    SELECT DISTINCT PATIENT_ID
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-07-31'
      AND (
            PROCEDURE_CODE IN ('J1743')     -- procedure
         OR NDC11 IN ('54092070001', '540920700')   -- ndc11
      )
 
    UNION
 
    -- Treatment in PHARMACY EVENTS table
    SELECT DISTINCT PATIENT_ID
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-07-31'
      AND NDC11 IN ('54092070001', '540920700')
)
 
-- Step 3: Final cohort meeting all 3 rules
SELECT
    COUNT(DISTINCT DX.PATIENT_ID) AS FINAL_PATIENT_COUNT
FROM DX_COUNTS DX
JOIN TREATMENT_PTS T
    ON DX.PATIENT_ID = T.PATIENT_ID
WHERE
      DX.E763_COUNT >= 2     -- ≥2 E763 Dx claims (5-year window)
  --AND --DX.E761_COUNT = 0;     -- 0 E761 Dx claims (5-year window)